In [0]:
bronze_path = "/Volumes/databricks-projects/databricks-schema/mock_bronze"


print("Bronze layer simulated successfully!")


Step 1: Create a Mock Raw Ingestion Stream (The Bronze Layer)

Run this code in a Databricks Notebook to populate a table with realistic CDC transaction log messages

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS
`databricks-projects`.`databricks-schema`.`mock_bronze`;

In [0]:
# 1. Simulate data that would be ingested from Lakeflow Connect
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType
from pyspark.sql.functions import current_timestamp

# Create data mimicking database operations: c=create, u=update, d=delete
mock_cdc_data = [
    # User 1 registers
    (1, "Alice", "Active", "c", 1001, 1717113600000), 
    # User 2 registers
    (2, "Bob", "Pending", "c", 1002, 1717113605000),  
    # User 1 updates their name
    (1, "Alice Smith", "Active", "u", 1003, 1717113610000), 
    # User 2 gets deleted from the source database
    (2, None, None, "d", 1004, 1717113620000) 
]

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("op", StringType(), True),            # Operation type
    StructField("scn", LongType(), True),             # Sequence number
    StructField("ts_ms", LongType(), True)            # Timestamp
])

# Create the data frame and write it as our raw Bronze layer
bronze_df = spark.createDataFrame(mock_cdc_data, schema)\
                 .withColumn("ingested_at", current_timestamp())

# Save it to a local workspace path
bronze_df.write.format("delta").mode("overwrite").save(bronze_path)
print("Bronze layer simulated successfully!")


In [0]:
%sql
SELECT *
FROM delta.`/Volumes/databricks-projects/databricks-schema/mock_bronze`;

In [0]:
display(
    spark.read.format("delta")
    .load("/Volumes/databricks-projects/databricks-schema/mock_bronze")
)

In [0]:
spark.sql("SHOW CATALOGS").show()


Step 2: Initialize the Target Silver Layer

The MERGE statement requires a target table to already exist. Run this to build an empty target table matching your clean master schema:

In [0]:
# Create an empty Delta table for your clean data
spark.sql("DROP TABLE IF EXISTS silver_users")

spark.sql("""
    CREATE TABLE silver_users (
        id INT,
        name STRING,
        status STRING,
        last_updated_scn LONG
    ) USING DELTA
""")
print("Silver table initialized!")



Step 3: Write the Production Deduplication & Merge Engine

This script reads your raw logs, identifies the absolute newest event for every record key (deduplication), and safely updates the master table.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
from delta.tables import DeltaTable

# 1. Read the newly arrived raw logs from the Bronze layer
raw_changes_df = spark.read.format("delta").load(bronze_path)

# 2. Micro-batch Deduplication
# If a record changed multiple times in the same batch, order by sequence number (scn) desc
window_spec = Window.partitionBy("id").orderBy(col("scn").desc())

latest_changes_df = raw_changes_df\
    .withColumn("rn", row_number().over(window_spec))\
    .filter("rn = 1")\
    .drop("rn")

# 3. Reference the target Silver table
silver_table = DeltaTable.forName(spark, "silver_users")

# 4. Perform the transactional ACID Merge
(silver_table.alias("target")
 .merge(latest_changes_df.alias("source"), "target.id = source.id")
 
 # Action A: If the database log says 'd', delete the row permanently
 .whenMatchedDelete(condition = "source.op = 'd'")
 
 # Action B: If it's an update, modify the records
 .whenMatchedUpdate(
     condition = "source.op != 'd' AND source.scn > target.last_updated_scn",
     set = {
         "name": "source.name",
         "status": "source.status",
         "last_updated_scn": "source.scn"
     }
 )
 
 # Action C: If the user doesn't exist yet, insert them
 .whenNotMatchedInsert(
     condition = "source.op != 'd'",
     values = {
         "id": "source.id",
         "name": "source.name",
         "status": "source.status",
         "last_updated_scn": "source.scn"
     }
 )
 .execute())

# 5. Review the final state
display(spark.sql("SELECT * FROM silver_users"))


Instead of loading the entire source dataset again, we will ingest only newly arrived records and apply the changes to the existing Silver table.

We will continue using your existing:
- Bronze Volume: /Volumes/databricks-projects/databricks-schema/mock_bronze
- Silver table: silver_users
- Business key: id
- Sequence number: scn

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType
)
from pyspark.sql.functions import current_timestamp

# Batch 2: New incremental CDC events
incremental_cdc_data = [
    (1, "Alice Johnson", "Active", "u", 1005, 1717113630000),
    (3, "Charlie", "Pending", "c", 1006, 1717113640000),
    (3, "Charlie Brown", "Active", "u", 1007, 1717113650000)
]




incremental_df = (
    spark.createDataFrame(incremental_cdc_data, schema)
    .withColumn("ingested_at", current_timestamp())
)

display(incremental_df)

We are reusing your original schema.
4.2 Append new records into Bronze

In [0]:
incremental_df.write \
    .format("delta") \
    .mode("append") \
    .save(bronze_path)

print("Incremental CDC batch appended successfully!")

In [0]:
%sql
SELECT *
FROM delta.`/Volumes/databricks-projects/databricks-schema/mock_bronze`
ORDER BY scn;

4.3 Read only the new incremental records
For this practice exercise, our previous batch ended at SCN 1004.

In [0]:
# Read Bronze Delta data
all_bronze_df = (
    spark.read.format("delta")
    .load(bronze_path)
)

# Read only newly arrived events
incremental_source_df = all_bronze_df.filter(
    col("scn") > 1004
)

display(incremental_source_df)

In [0]:
from pyspark.sql.functions import col

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = (
    Window
    .partitionBy("id")
    .orderBy(col("scn").desc())
)

deduplicated_df = (
    incremental_source_df
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

display(deduplicated_df)

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(spark, "silver_users")

(
    silver_table.alias("target")
    .merge(
        deduplicated_df.alias("source"),
        "target.id = source.id"
    )
    .whenMatchedDelete(
        condition="""
            source.op = 'd'
            AND source.scn > target.last_updated_scn
        """
    )
    .whenMatchedUpdate(
        condition="source.scn > target.last_updated_scn",
        set={
            "name": "source.name",
            "status": "source.status",
            "last_updated_scn": "source.scn"
        }
    )
    .whenNotMatchedInsert(
        condition="source.op IN ('c', 'u')",
        values={
            "id": "source.id",
            "name": "source.name",
            "status": "source.status",
            "last_updated_scn": "source.scn"
        }
    )
    .execute()
)

print("Incremental MERGE completed successfully!")

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM silver_users
        ORDER BY id
    """)
)

In [0]:
print("Bronze total CDC events:",
      all_bronze_df.count())

print("Silver current active users:",
      spark.table("silver_users").count())